# Auto-Tuning GPU kernels in Julia

GPU kernels have many parameters that do not change *what* they compute, but do change *how fast* they compute it: the size and shape of the workgroups, how much work each work-item does, the precision of the data, and so on. The best values depend on the GPU, the problem size and the kernel itself, and they are hard to predict: the fastest configuration on one GPU is often not the fastest on another. **Auto-tuning** finds them by benchmarking many configurations and picking the best one.

In this notebook we use [Kernel Tuner](https://kerneltuner.github.io/kernel_tuner/stable/) to auto-tune a KernelAbstractions.jl kernel. Kernel Tuner is a Python package; [KernelTuner.jl](https://github.com/KernelTuner/KernelTuner.jl) lets you use it from Julia. You write the kernel once, mark which values are tunable, and Kernel Tuner compiles, runs and times every configuration for you.

If you're interested in the kernel we're using, the [KernelAbstractions Gray-Scott tutorial](https://github.com/JuliaParallel/julia-hpc-tutorial-sc26/tree/main/02-Parallel-Programming/KernelAbstractions) does a great job explaining the basic GPU programming concepts involved.

**Overview**
1. **The Gray-Scott model:** the kernel we will tune, and how to make its parameters tunable.
2. **Auto-tuning the kernel:** tuning the workgroup size and the tile size, and reading the results.
3. **Expanding the tuning:** restrictions, optimization strategies, limits and caching, for search spaces that are too large to try exhaustively.
4. **Advanced tuning options:** verifying the output, measuring energy, mixed precision and user-defined metrics.
5. **All combined:** tuning for both time and energy, and choosing a configuration from the Pareto front.

**How to use this notebook.** The kernels are given: you wrote them in the KernelAbstractions tutorial. The exercises are about tuning them. In the code cells, fill in the blanks marked `___`; a `# TODO` comment next to each blank says what goes there. Julia refuses to run a cell that still contains a `___`, so you cannot skip one by accident. Some exercises also ask questions about your results. Proposed solutions, with a brief explanation, are in the `proposed_solution` folder, one file per exercise.

# Setup

The cells below activate the project environment, install and load the packages, and select the backend. You don't need to tell the notebook which GPU you have: [GPUSelect.jl](https://github.com/SimonDanisch/GPUSelect.jl) detects it, and the first time `setup_notebook.jl` installs the matching backend package (CUDA, AMDGPU, Metal or oneAPI).

`setup_notebook.jl` also defines the helper function `get_cell_by_tag(tag)`, which returns the source code of the notebook cell with that tag (see Section 2).

Because Plots is loaded, KernelTuner.jl also provides a few plotting functions that we use throughout this notebook, so that we can focus on the tuning rather than on plotting code:
- `plot_tuning_heatmap(results, x, y; objective="time", higher_is_better=false)` plots a heatmap of the objective over two tunable parameters;
- `with_live_tuning_plot(f, cachefile; ...)` runs the tuning in `f` while plotting the results in the cache file as they come in (see Section 3.3);
- `plot_time_energy` and `pareto_front` are used in Section 5 to plot time versus energy.

In [ ]:
# When running on a server, check that you are on a compute node
println("Running on node: ", gethostname())

In [ ]:
# Plots follow your system's light or dark mode automatically; uncomment to force light (false) or dark (true) plots
# dark_mode_override = false

In [ ]:
include("setup_notebook.jl")

Let's check the packages are installed correctly and set the backend we'll work with.

In [ ]:
Pkg.status()

In [ ]:
# GPUSelect loads the backend package that matches your GPU; fallback=false makes it an error to not have a GPU
Backend = GPUSelect.Backend(fallback=false)   # CUDABackend(), ROCBackend(), MetalBackend() or oneAPIBackend()
ArrayKA = GPUSelect.Storage(fallback=false)   # CuArray, ROCArray, MtlArray or oneArray
println("Backend: ", Backend, "\nArray type: ", ArrayKA)

# 1. The Gray-Scott Reaction-Diffusion Model

The Gray-Scott Reaction-Diffusion model is a mathematical model used to describe how two interacting substances, typically denoted as $U$ and $V$, evolve over time in a spatial domain. This model captures the combined effects of chemical reactions and diffusion, leading to complex pattern formations such as spots, stripes, and waves, which resemble natural phenomena like animal skin patterns or chemical reactions in a petri dish.

$$\frac{\partial U}{\partial t} = D_u \nabla^2 U - UV^2 + F(1-U), \qquad \frac{\partial V}{\partial t} = D_v \nabla^2 V + UV^2 - (F+k)V$$

- $D_u$ and $D_v$ (`D_u`, `D_v`) are the diffusion rates of $U$ and $V$;
- $UV^2$ is the reaction, which converts $U$ into $V$;
- $F$ (`F`) is the feed rate at which $U$ is replenished, and $k$ (`k`) the kill rate at which $V$ is removed.

We solve this on an `nx` × `ny` grid with grid spacing `dx`, approximate the Laplacian $\nabla^2$ with a 5-point stencil (each point and its four neighbors), and take explicit time steps of size `dt`. The optional `noise` adds random perturbations to $U$.

## The Kernel

The kernel below computes one time step; for more information on this implementation, see [the KernelAbstractions tutorial](../../KernelAbstractions/gray-scott.ipynb). Each work-item updates one grid point `(x, y)`:
- it reads the old values from `U` and `V`, and writes the new values to separate arrays `Uout` and `Vout`, because a work-item must not overwrite a value that its neighbors still need to read. The simulation swaps the input and output arrays after every step (double buffering);
- `@Const` tells the compiler that the inputs are read-only;
- the boundary points are skipped (`1 < x < nx`), so they keep their initial values.

## Making the Kernel Tunable

To tune a value, Kernel Tuner must be able to set it from outside the kernel. KernelTuner.jl does this by passing **every tunable parameter as an extra `Val` argument** at the end of the kernel call, in the order in which the parameters are listed in `tune_params` (Section 2.1). Only the position counts, not the name: the first tunable parameter, `block_size_x`, arrives as `BX`. With `::Val{BX}` and `where {BX}`, the value becomes a compile-time constant inside the kernel: Kernel Tuner compiles a specialized kernel for each configuration, so the compiler can, for example, unroll loops or size private arrays with it (we will use this in Section 2.2). Providing a default, `= Val(16)`, keeps the kernel usable outside of autotuning.

Here we add slots for the workgroup size `BX` × `BY`. The kernel body does not use them, because Kernel Tuner sets the workgroup size itself when it launches the kernel. But as Kernel Tuner passes *all* tunable parameters to the kernel, the signature needs a slot for each of them.

**Terminology.** The GPU vendors each use different names for the same concepts. In this notebook we use the KernelAbstractions terms, but Kernel Tuner names the workgroup dimensions after CUDA's thread blocks:

| KernelAbstractions | CUDA | AMD (HIP) | Apple Metal | Kernel Tuner |
|---|---|---|---|---|
| work-item | thread | work-item | thread | |
| workgroup | thread block | workgroup | threadgroup | `block_size_x`, `block_size_y`, `block_size_z` |
| (executed together in hardware) | warp, 32 threads | wavefront, 64 or 32 work-items | SIMD-group, 32 threads | |

In [ ]:
using KernelAbstractions

# 5-point Laplacian, same as the reference CPU implementation
@inline function laplacian(A, x, y, dx)
    @inbounds DAx = (A[x-1, y] - (2 * A[x, y]) + A[x+1, y]) / dx
    @inbounds DAy = (A[x, y-1] - (2 * A[x, y]) + A[x, y+1]) / dx
    return DAx + DAy
end

# BX × BY is the workgroup size: Kernel Tuner passes each tunable parameter to the kernel as a `Val` argument
@kernel function gs_gpu_kernel!(Uout, Vout, @Const(U), @Const(V), D_u, D_v, F, k, dt, dx, nx, ny, noise,
        ::Val{BX} = Val(16), ::Val{BY} = Val(16)) where {BX, BY}
    x, y = @index(Global, NTuple)

    if (1 < x < nx) && (1 < y < ny)
        u = @inbounds U[x, y]
        v = @inbounds V[x, y]

        ΔU = laplacian(U, x, y, dx)
        ΔV = laplacian(V, x, y, dx)
        UVV = u * v * v
        Nr = noise == 0 ? zero(u) : noise * randn(Float32)
        DU = D_u * ΔU - UVV + F * (1 - u) + Nr
        DV = D_v * ΔV + UVV - (F + k) * v
        newU = u + DU * dt
        newV = v + DV * dt

        @inbounds Uout[x, y] = newU
        @inbounds Vout[x, y] = newV
    end
end

Now that we have our kernel source code, it might be nice to see what it actually does. Don't worry about the implementation details of the cell below, they're just to draw a nice animation. 


In [ ]:
function simulate_gs_gpu(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                        init_multi = 0.5,
                        noise = 0.0f0,
                        D_u = 0.16f0,
                        D_v = 0.08f0,
                        F = 0.035f0,
                        k = 0.06f0,
                        dt = 1.0f0,
                        dx = 1.0f0,
                        plot_every = 500)

    U = Float32.(rand(0.0:0.001:0.1, n, n))
    V = zeros(Float32, n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(init_multi)
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(0.5 * init_multi)

    U1 = KernelAbstractions.allocate(Backend, Float32, n, n)
    V1 = KernelAbstractions.allocate(Backend, Float32, n, n)
    U2 = KernelAbstractions.allocate(Backend, Float32, n, n)
    V2 = KernelAbstractions.allocate(Backend, Float32, n, n)
    KernelAbstractions.copyto!(Backend, U1, U)
    KernelAbstractions.copyto!(Backend, V1, V)
    KernelAbstractions.copyto!(Backend, U2, U)
    KernelAbstractions.copyto!(Backend, V2, V)

    kernel = gs_gpu_kernel!(Backend, (16, 16))   # workgroup size of 16 × 16 work-items
    readU, readV, writeU, writeV = U1, V1, U2, V2

    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model", xlabel="X", ylabel="Y")

    anim = Plots.@animate for t in 1:div(nsteps, plot_every)
        for _ in 1:plot_every
            kernel(writeU, writeV, readU, readV, D_u, D_v, F, k, dt, dx, n, n, noise, ndrange=(n, n))
            readU, readV, writeU, writeV = writeU, writeV, readU, readV
        end

        KernelAbstractions.copyto!(Backend, U, readU)
        KernelAbstractions.synchronize(Backend)

        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model", dpi=110)
    end
    gif(anim, "gs_ka.gif"; fps=10)
end

Let's see our Gray-Scott Model in action. This runs 10,000 time steps on a 500 × 500 grid and plots every 500th step: starting from a small square in the middle of the grid, you should see patterns grow and spread over the grid. The first run also compiles the kernel, so it takes longer than following runs.

In [ ]:
@time simulate_gs_gpu(500, 10000)

# 2. Autotuning the Kernel

We can see the kernel works. But is it as fast as it can be? In this section we first tune the workgroup size that we just made tunable, and then a second kernel with an additional optimization.

**What Kernel Tuner does.** For each configuration in the search space, Kernel Tuner compiles the kernel with those parameter values, runs it several times (7 by default, which you can change with the `iterations` argument), and reports the average time in milliseconds. This is the time the kernel runs on the GPU, without the time it takes to launch the kernel from the host. As every configuration is compiled separately, tuning takes much longer than the sum of the reported times.

**How Kernel Tuner gets the kernel.** Kernel Tuner compiles the kernel itself, in a separate module, for every configuration. Therefore it needs the *source code* of the kernel, not the Julia function we defined above. Normally, you would pass the name of a file with the kernel or a string with its source code. In this notebook, the kernel cells carry a *tag* in their cell metadata (e.g. `gs_gpu_kernel`), and the helper `get_cell_by_tag` reads the source code of the tagged cell from the notebook file. Because it reads the file on disk, **save the notebook after changing a kernel cell**, otherwise Kernel Tuner tunes the previous version.

We start by generating the input data. Kernel Tuner runs the kernel with these arrays, so they must be allocated on the GPU.

In [ ]:
# parameters
n = 1024
init_multi = 0.5

U = Float32.(rand(0.0:0.001:0.1, n, n))
V = zeros(Float32, n, n)

n_mid = div(n, 2)
U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(1.0 * init_multi)
V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(0.5 * init_multi)

U1 = KernelAbstractions.allocate(Backend, Float32, n, n)
V1 = KernelAbstractions.allocate(Backend, Float32, n, n)
U2 = KernelAbstractions.allocate(Backend, Float32, n, n)
V2 = KernelAbstractions.allocate(Backend, Float32, n, n)
KernelAbstractions.copyto!(Backend, U1, U)
KernelAbstractions.copyto!(Backend, V1, V)
KernelAbstractions.copyto!(Backend, U2, U)
KernelAbstractions.copyto!(Backend, V2, V)

## 2.1. Tuning the workgroup size

We start with the simple kernel `gs_gpu_kernel!`, where every work-item computes exactly one grid point. The only thing we can tune here is the **workgroup size**: how many work-items are grouped together, and in what shape (`block_size_x` × `block_size_y`).

We call `KernelTuner.tune_kernel` with the following arguments:
1. the name of the kernel function, `"gs_gpu_kernel!"`;
2. the source code of the kernel, see above;
3. the problem size `(n, n)`: the number of work-items needed in each dimension;
4. the kernel arguments: the arrays and values that Kernel Tuner calls the kernel with. These must match the kernel signature in order *and* in type: for example, `0.16f0` is a `Float32`, as the kernel computes in 32-bit precision. The tunable parameters are appended by Kernel Tuner, so they are not part of this list;
5. the tunable parameters: a vector of `(name, values)` tuples. This is a vector rather than a `Dict`, because the order matters: it is the order of the `Val` arguments in the kernel signature.

These are followed by keyword arguments, such as `verbose=true` to print more information during tuning.

Kernel Tuner uses the tunable parameters named `block_size_x` and `block_size_y` as the workgroup dimensions (use the `block_size_names` argument for other names). By default, it computes the number of workgroups by dividing the problem size by the workgroup size, rounded up: `cld(n, block_size_x)` × `cld(n, block_size_y)` workgroups. In Section 2.2 we will see how to change this.

In [ ]:
using KernelTuner

arguments = [U1, V1, U2, V2, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0]
tune_params = [
    ("block_size_x", [8, 16, 32, 64, 128]),  # workgroup size along x
    ("block_size_y", [1, 2, 4, 8, 16]),      # workgroup size along y
]

results_wg, env_wg = KernelTuner.tune_kernel(
    "gs_gpu_kernel!",
    get_cell_by_tag("gs_gpu_kernel"),   # we're getting the kernel source code from this notebook, but normally you would just pass either the string or the filename
    (n, n),
    arguments,
    tune_params,
    verbose=true,
)

best_wg = (env_wg["best_config"]["block_size_x"], env_wg["best_config"]["block_size_y"])

Kernel Tuner prints the time of each configuration as it evaluates them. `tune_kernel` also returns two values:
- `results_wg`, a vector with one `Dict` per evaluated configuration, containing the values of the tunable parameters, the average `time` in milliseconds, the individual measurements in `times`, and more;
- `env_wg`, a `Dict` describing the environment in which the tuning took place, such as the device name, the tuning options and `best_config`, the best configuration found. It is good practice to store this information along with your results.

In [ ]:
display(results_wg)
display(env_wg)

In [ ]:
plot_tuning_heatmap(results_wg, "block_size_x", "block_size_y")

The heatmap shows the time of each configuration (lower is better). Is the fastest workgroup shape square, or wide in one direction? Julia arrays are column-major: `U[x, y]` and `U[x+1, y]` are next to each other in memory. So with a larger `block_size_x`, neighboring work-items read neighboring memory addresses, which the GPU can combine into fewer memory transactions (*coalescing*).

What's going on with the missing configuration for `(128, 16)`? Kernel Tuner queries the device for the maximum number of work-items per workgroup, which is 1024 on most GPUs, and skips configurations that exceed it: 128 × 16 = 2048. Configurations that fail to compile or launch are skipped as well; with `verbose=true`, Kernel Tuner reports why. In Section 3.1 we will add restrictions of our own.

## 2.2. Tuning the tile size

Besides the workgroup size, we can also tune how the kernel does its work. In `gs_gpu_kernel!`, each work-item reads its own point and its four neighbors, of both `U` and `V`, and neighboring work-items read the same values again: each value is read five times. With **tiling**, each work-item computes a tile of `Q` × `P` grid points (`Q` along `x`, `P` along `y`). It first loads the tile plus a border of one point (the *halo*) into private memory, i.e. registers, and then computes all points of the tile from there: $(Q+2)(P+2)$ reads for $QP$ points, instead of $5QP$. Larger tiles reuse more data, but also need more registers. When a work-item needs more registers than available, values spill to slower memory, and fewer work-items can run at the same time. Where the optimum lies is hard to predict, which makes it a good job for auto-tuning.

The kernel `gs_gpu_kernel_tiled!` below implements this in four sections: compute the first grid point of the tile `(x1, y1)`, load the tiles of `U` and `V` with `load_tile!`, compute the new values into the private arrays `stepU` and `stepV`, and write these to the output arrays. `@private Float32 (Q + 2, P + 2)` allocates a private array for each work-item. Its size must be known at compile time, which is why `P` and `Q` are `Val` parameters.

The kernel has four tunable parameters, `P`, `Q`, `BX` and `BY`, in the same order as in `tune_params` below. As each workgroup now covers `(block_size_x * Q) × (block_size_y * P)` grid points, fewer workgroups are needed. We tell Kernel Tuner this with `grid_div_x` and `grid_div_y`: the number of workgroups in a dimension is the problem size divided by the product of the listed parameters, rounded up, e.g. `cld(n, block_size_x * Q)` along `x`.

In [ ]:
using KernelAbstractions

# Fill per-work-item halo cache tile[q, p] = A[x1+q-2, y1+p-2] (out-of-bounds → zero).
# tile is `@private Float32 (Q + 2, P + 2)` — compile-time size from kernel parameters P, Q.
@inline function load_tile!(tile, A, x1, y1, nx, ny, ::Val{P}, ::Val{Q}) where {P, Q}
    @inbounds for q in 1:(Q + 2), p in 1:(P + 2)
        x = x1 + q - 2
        y = y1 + p - 2
        if 1 <= x <= nx && 1 <= y <= ny
            tile[q, p] = A[x, y]
        else
            tile[q, p] = zero(eltype(A))
        end
    end
    return
end

# Laplacian at tile indices (q, p) in 1:Q, 1:P — same 5-point formula as the CPU reference.
@inline function laplacian_tile(tile, q, p, dx)
    u = tile[q + 1, p + 1]
    @inbounds ux_m = tile[q, p + 1]
    @inbounds ux_p = tile[q + 2, p + 1]
    @inbounds uy_m = tile[q + 1, p]
    @inbounds uy_p = tile[q + 1, p + 2]
    ((ux_m - 2 * u + ux_p) + (uy_m - 2 * u + uy_p)) / dx
end

@kernel function gs_gpu_kernel_tiled!(Uout, Vout, @Const(U), @Const(V),
        D_u, D_v, F, k, dt, dx, nx, ny, noise,
        ::Val{P} = Val(1), ::Val{Q} = Val(1), ::Val{BX} = Val(16), ::Val{BY} = Val(16)) where {P, Q, BX, BY}
    # Section 1: get indices
    i, j = @index(Global, NTuple)
    x1 = (i - 1) * Q + 1
    y1 = (j - 1) * P + 1

    # Section 2: allocate register memory and get data
    tileU = @private Float32 (Q + 2, P + 2)
    tileV = @private Float32 (Q + 2, P + 2)
    stepU = @private Float32 (Q, P)
    stepV = @private Float32 (Q, P)

    if (x1 < nx) && (x1 + Q > 2)
        load_tile!(tileU, U, x1, y1, nx, ny, Val(P), Val(Q))
        load_tile!(tileV, V, x1, y1, nx, ny, Val(P), Val(Q))

        # Section 3: calculate Laplacians (values → private step tile)
        for q in 1:Q, p in 1:P
            if (1 < x1 + q - 1 < nx) && (1 <  y1 + p - 1 < ny)
                u = tileU[q + 1, p + 1]
                v = tileV[q + 1, p + 1]
                ΔU = laplacian_tile(tileU, q, p, dx)
                ΔV = laplacian_tile(tileV, q, p, dx)
                UVV = u * v * v
                Nr = noise == 0 ? zero(u) : noise * randn(Float32)
                DU = D_u * ΔU - UVV + F * (1 - u) + Nr
                DV = D_v * ΔV + UVV - (F + k) * v
                stepU[q, p] = u + DU * dt
                stepV[q, p] = v + DV * dt
            end
        end

        # Section 4: global writes
        for q in 1:Q, p in 1:P
            x = x1 + q - 1
            y = y1 + p - 1
            if (1 < x < nx) && (1 < y < ny)
                @inbounds Uout[x, y] = stepU[q, p]
                @inbounds Vout[x, y] = stepV[q, p]
            end
        end
    end
end

First, we tune only the tile size `P` × `Q`, keeping the best workgroup size from Section 2.1 fixed. This way we tune one group of parameters at a time; in Section 3.3 we will see whether tuning everything at once finds something better.

In [ ]:
using KernelTuner

arguments = [U1, V1, U2, V2, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0]
tune_params = [
    ("P", [1, 2, 4, 8, 16]),                    # tile size per work-item along y
    ("Q", [1, 2, 4, 8, 16]),                    # tile size per work-item along x
    ("block_size_x", [best_wg[1]]),             # keep the best workgroup size from step 1 fixed
    ("block_size_y", [best_wg[2]]),
]

results_ti, env_ti = KernelTuner.tune_kernel(
    "gs_gpu_kernel_tiled!",
    get_cell_by_tag("gs_gpu_kernel_tiled"), # we're getting the kernel source code from this notebook, but normally you would just pass either the string or the filename
    (n, n),
    arguments,
    tune_params,
    # each workgroup covers (block_size_x * Q) × (block_size_y * P) grid points
    grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
    verbose=true
)

display(results_ti)

In [ ]:
plot_tuning_heatmap(results_ti, "P", "Q")

Which tile size is the fastest, and is it faster than the best configuration without tiling from Section 2.1 (compare the best times printed by Kernel Tuner)? Also look at the largest tiles: with `P = Q = 16`, each work-item keeps four private arrays of up to 18 × 18 values, far more than fits in its registers.

# 3. Expanding the Tuning

So far, we tuned a few parameters at a time, with a few values each. In this section we tune all four parameters of the tiled kernel together. The search space then quickly becomes too large to try every configuration, so we first look at the tools Kernel Tuner offers for this: restrictions, optimization strategies, limits and caching. In Section 3.3 we put them all together.

## 3.1. Restrictions

Some configurations are not worth trying, because they can not work or are always going to be slow. With Kernel Tuner's `restrictions` argument, we can remove these from the search space before tuning starts. It takes a list of expressions, as strings, in terms of the tunable parameters. Only the configurations for which all expressions are true remain in the search space.

Here we restrict the total workgroup size. The upper bound, 1024 work-items, is the hardware maximum on most GPUs. The lower bound comes from how GPUs execute work-items: in groups of 32 (NVIDIA warps, Apple SIMD-groups) or 64 (AMD wavefronts on data center GPUs). A smaller workgroup leaves part of such a group idle. We define the restrictions here and use them in Section 3.3.

In [ ]:
# keep workgroups between one SIMD-group (32 work-items) and the hardware maximum (1024 work-items)
restrictions = ["32 <= block_size_x * block_size_y <= 1024"]

## 3.2 Optimization Strategies & Limits

By default, Kernel Tuner tries every configuration in the search space ("brute force"). This was fine for the search spaces so far, but the size of the search space is the product of the number of values of each parameter. In Section 3.3, four parameters with five values each give 5⁴ = 625 configurations. At about a second each to compile and benchmark, that takes over ten minutes, and real kernels often have many more parameters.

Optimization strategies instead evaluate only part of the search space, choosing which configuration to try next based on the results so far. Kernel Tuner provides a wide range of [optimization strategies](https://kerneltuner.github.io/kernel_tuner/stable/optimization.html#optimization-strategies), for example:
- `random_sample` evaluates a random selection of configurations, a simple baseline;
- `genetic_algorithm` evolves a population of configurations by combining and mutating the best ones;
- `dual_annealing` combines simulated annealing, which can escape local optima, with local search;
- `bayes_opt` (Bayesian optimization) builds a statistical model of the search space to predict which configurations are promising.

Most strategies are stochastic: they may find a different result in each run, and are not guaranteed to find the best configuration.

In [ ]:
strategy = "genetic_algorithm"
# strategy = "dual_annealing"
# strategy = "random_sample"
# strategy = "bayes_opt"

In addition, a strategy needs to know when to stop. We can set a limit on the number of configurations to evaluate with `max_fevals` ("function evaluations"), or on the tuning time in seconds with `time_limit`. When the limit is hit, Kernel Tuner stops and returns the best configuration found so far. Limits, and any strategy-specific options such as the population size of the genetic algorithm, are passed to `strategy_options` as a vector of `(name, value)` tuples.

In [ ]:
# strategy_options=[("max_fevals", 100)]
strategy_options=[("time_limit", 30)]

## 3.3. Caching & Combining It All

The best tile size may depend on the workgroup size and vice versa, so finally we tune both at the same time, using the restrictions, strategy and limit defined above. First, let's see how much larger the search space has become:

In [ ]:
tune_params = [
    ("P", [1, 2, 4, 8, 16]),                    # tile size per work-item along y
    ("Q", [1, 2, 4, 8, 16]),                    # tile size per work-item along x
    ("block_size_x", [8, 16, 32, 64, 128]),     # workgroup size along x
    ("block_size_y", [1, 2, 4, 8, 16]),         # workgroup size along y
]

# the number of configurations, before and after applying our restriction on the workgroup size
all_configs = Iterators.product(last.(tune_params)...)
println("Configurations: ", length(all_configs))
println("Configurations that satisfy the restrictions: ", count(c -> 32 <= c[3] * c[4] <= 1024, all_configs))

As tuning now takes longer, it is convenient to save the results in a *cache file*, by passing a filename of our choice with the `cache` argument. Kernel Tuner writes each configuration to the cache file as soon as it has been measured. When tuning is interrupted, or when you run the cell again, configurations that are already in the cache are read from it instead of being measured again, so you can pick up where you left off. The cache file is also a record of your results that you can analyze later.

While tuning, the helper `with_live_tuning_plot` reads the cache file in the background and keeps a scatterplot of the results so far up to date. Try running the cell a second time: the configurations from the first run are now read from the cache, so the strategy gets further within the same time limit.

In [ ]:
using KernelTuner

arguments = [U1, V1, U2, V2, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0]
cachefile = "gs_gpu_kernel_tuned.json"

# the plot above the tuning output is redrawn as configurations are added to the cache file
results_wg_ti, env_wg_ti = with_live_tuning_plot(cachefile) do
    KernelTuner.tune_kernel(
        "gs_gpu_kernel_tiled!",
        get_cell_by_tag("gs_gpu_kernel_tiled"),
        (n, n),
        arguments,
        tune_params,
        restrictions=restrictions,
        # each workgroup covers (block_size_x * Q) × (block_size_y * P) grid points
        grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
        verbose=true,
        cache=cachefile,
        strategy=strategy,
        strategy_options=strategy_options,
    )
end

# if you'd like to run the cell again without the cached results, uncomment the following line
# rm(cachefile)

How does tuning everything at once compare to tuning one group of parameters at a time, as in Section 2? Keep in mind that the strategy has only evaluated part of the search space within its time limit.

In [ ]:
println("2.1, workgroup size only:            ", round(env_wg["best_config"]["time"], digits=3), " ms")
println("2.2, tile size with best workgroup:  ", round(env_ti["best_config"]["time"], digits=3), " ms")
println("3.3, all together ($(length(results_wg_ti)) configurations): ", round(env_wg_ti["best_config"]["time"], digits=3), " ms")
env_wg_ti["best_config"]

# 4. Advanced Tuning Options

Execution time is not the only thing worth tuning for. In this section we look at some more advanced features of Kernel Tuner: checking that every configuration still computes the right result, measuring energy, trading accuracy for speed with mixed precision, and defining your own metrics.

To keep each example quick, we use a smaller search space than in Section 3.

In [ ]:
# a smaller search space, so that each example below only takes a few seconds
tune_params_small = [
    ("P", [1, 2, 4]),                   # tile size per work-item along y
    ("Q", [1, 2, 4]),                   # tile size per work-item along x
    ("block_size_x", [32, 64, 128]),    # workgroup size along x
    ("block_size_y", [1, 2, 4]),        # workgroup size along y
]

tune_params_extra_small = [
    ("P", [1, 2]),                   # tile size per work-item along y
    ("Q", [1, 2]),                   # tile size per work-item along x
    ("block_size_x", [32, 64]),    # workgroup size along x
    ("block_size_y", [1, 2]),        # workgroup size along y
]

arguments = [U1, V1, U2, V2, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0]

## 4.1. Output Verification / Provided Answers

A fast kernel is useless if it computes the wrong result. Tiling makes the indexing more complex, so it is easy to introduce a bug that only shows up for some tile sizes. Kernel Tuner can check the output of every configuration against a reference: pass the expected output with the `answer` argument. This is a list with one entry per kernel argument: the expected array for each output you want to check, and `nothing` for the arguments to skip.

As reference, we run one step of the simple kernel `gs_gpu_kernel!` on the CPU. Thanks to KernelAbstractions, this is the same kernel code, just launched on a different backend. The reference output arrays start as copies of the input, just like `U1` and `V1`, because the kernel does not write the boundary cells.

In [ ]:
# reference output: one step of the simple kernel, run on the CPU
U_ref, V_ref = copy(U), copy(V)
gs_gpu_kernel!(CPU(), (16, 16))(U_ref, V_ref, U, V, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0, ndrange=(n, n))

# one entry per kernel argument: check Uout and Vout, skip the rest
answer = [U_ref, V_ref, fill(nothing, length(arguments) - 2)...]

results_verified, _ = KernelTuner.tune_kernel(
    "gs_gpu_kernel_tiled!",
    get_cell_by_tag("gs_gpu_kernel_tiled"),
    (n, n),
    arguments,
    tune_params_extra_small,
    restrictions=restrictions,
    grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
    answer=answer,
    atol=1e-5,  # maximum absolute difference allowed, the GPU and CPU may round slightly differently
    verbose=true,
)

println("All $(length(results_verified)) configurations computed the correct result.")

If a configuration computes a different result, Kernel Tuner stops with an error such as `Kernel result verification failed for: P=1, Q=1, block_size_x=32, block_size_y=1`, so you know exactly which configuration to debug. The default check requires all values to be within `atol` of the answer. For other checks, you can pass your own function with the `verify` argument: `verify(answer, result; atol)` should return `true` or `false`.

## 4.2. Energy Tuning

On many systems, energy is as much a constraint as time. Kernel Tuner measures energy with *observers*: objects that collect extra measurements while a configuration is benchmarked. `KernelTuner.get_energy_observer_for_backend` returns the right observer for your GPU (NVML on NVIDIA, AMD SMI on AMD, `powermetrics` on Apple), together with the names of the quantities it measures.

We pass the observer with `observers`, and set `objective` to the energy so that Kernel Tuner reports the most energy-efficient configuration instead of the fastest one. As lower energy is better, we set `objective_higher_is_better=false`. Measuring energy takes longer, as the observer measures each configuration for about a second to get a reliable reading, so we use the small search space.

<div class="alert alert-info">
  <strong>Apple Silicon</strong><br />Reading the GPU energy with <tt>powermetrics</tt> requires administrator rights. To measure without a password prompt, run <tt>sudo visudo</tt> and add the line <tt>your_username ALL=(ALL) NOPASSWD: /usr/bin/powermetrics</tt>.
</div>

In [ ]:
# "default" selects the observer for the first GPU backend that Julia finds
energy_observer, energy_fields = KernelTuner.get_energy_observer_for_backend("default")
energy_key = only(filter(endswith("energy"), energy_fields))   # the name differs per vendor, e.g. "energy" or "nvml_energy"
println("Measuring: ", join(energy_fields, ", "))

results_energy, _ = KernelTuner.tune_kernel(
    "gs_gpu_kernel_tiled!",
    get_cell_by_tag("gs_gpu_kernel_tiled"),
    (n, n),
    arguments,
    tune_params_small,
    restrictions=restrictions,
    grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
    observers=[energy_observer],
    objective=energy_key,   # optimize for energy instead of time
    objective_higher_is_better=false,   # lower energy is better
    verbose=true,
)

Let's compare the time and energy of each configuration, and see whether the fastest configuration is also the most energy-efficient one:

In [ ]:
config_string(r) = join(["$k=$(r[k])" for k in first.(tune_params_small)], ", ")

times = [r["time"] for r in results_energy]
energies = [r[energy_key] for r in results_energy]
fastest, most_efficient = argmin(times), argmin(energies)
println("Fastest:       ", config_string(results_energy[fastest]), "  ($(round(times[fastest], digits=3)) ms, $(round(energies[fastest], sigdigits=3)) J)")
println("Lowest energy: ", config_string(results_energy[most_efficient]), "  ($(round(times[most_efficient], digits=3)) ms, $(round(energies[most_efficient], sigdigits=3)) J)")

scatter(times, energies; label="configuration", marker_z=energies, color=reverse(cgrad(:RdYlGn)), colorbar=false,
    markersize=5, markerstrokewidth=0.3, markerstrokecolor=:gray40,
    xlabel="Time (ms)", ylabel="Energy (J)", title="Time versus energy", size=(800, 400), dpi=200,
    left_margin=3Plots.mm, bottom_margin=3Plots.mm)
scatter!([times[fastest]], [energies[fastest]]; marker=:star5, markersize=10, color=:dodgerblue, label="fastest")
scatter!([times[most_efficient]], [energies[most_efficient]]; marker=:diamond, markersize=8, color=:magenta, label="lowest energy")

Time and energy are related, as a kernel that finishes sooner draws power for a shorter time. But they are not the same: a configuration that keeps the GPU less busy can be slightly slower, yet draw less power. Keep in mind that energy is measured with a coarser resolution than time, especially for short kernels like this one, so expect some noise in these measurements.

## 4.3. Accuracy Tuning & Mixed Precision

So far, every configuration had to produce (nearly) the same result. Sometimes, a less accurate result is acceptable if it is computed faster. A common example is *mixed precision*: storing data or doing arithmetic in 16-bit instead of 32-bit floating point halves the memory traffic and can use faster hardware, at the cost of accuracy.

We make a variant of the simple kernel with two extra tunable parameters: `storage_bits`, the precision of the arrays, and `compute_bits`, the precision of the arithmetic. Julia kernels are generic, so the kernel works for any array type; the compute type `T` is chosen from the `COMPUTE_BITS` parameter, and values are converted to `T` before any arithmetic.

Note that not every GPU computes faster in 16-bit: support for and speed of `Float16` arithmetic differ per GPU and backend.

In [ ]:
using KernelAbstractions

# 5-point Laplacian like before, but with the values converted to the compute type `T` before the arithmetic
@inline function laplacian_T(::Type{T}, A, x, y, dx) where {T}
    @inbounds c = T(A[x, y])
    @inbounds DAx = (T(A[x-1, y]) - 2 * c + T(A[x+1, y])) / T(dx)
    @inbounds DAy = (T(A[x, y-1]) - 2 * c + T(A[x, y+1])) / T(dx)
    return DAx + DAy
end

# Same computation as `gs_gpu_kernel!`, with two extra tunable parameters:
# STORAGE_BITS is the precision of the arrays (Kernel Tuner passes the matching arrays, see below), COMPUTE_BITS the precision of the arithmetic.
@kernel function gs_gpu_kernel_mixed!(Uout, Vout, @Const(U), @Const(V), D_u, D_v, F, k, dt, dx, nx, ny, noise,
        ::Val{BX} = Val(16), ::Val{BY} = Val(16), ::Val{STORAGE_BITS} = Val(32), ::Val{COMPUTE_BITS} = Val(32)) where {BX, BY, STORAGE_BITS, COMPUTE_BITS}
    T = COMPUTE_BITS == 16 ? Float16 : Float32
    x, y = @index(Global, NTuple)

    if (1 < x < nx) && (1 < y < ny)
        u = @inbounds T(U[x, y])
        v = @inbounds T(V[x, y])

        ΔU = laplacian_T(T, U, x, y, dx)
        ΔV = laplacian_T(T, V, x, y, dx)
        UVV = u * v * v
        DU = T(D_u) * ΔU - UVV + T(F) * (1 - u)
        DV = T(D_v) * ΔV + UVV - (T(F) + T(k)) * v

        @inbounds Uout[x, y] = u + DU * T(dt)   # converted to the storage type on write
        @inbounds Vout[x, y] = v + DV * T(dt)
    end
end

Depending on `storage_bits`, Kernel Tuner must pass either the 32-bit or the 16-bit arrays to the kernel. For this we use `KernelTuner.Tunable(param, Dict(value => array))`: for each configuration, Kernel Tuner selects the array that matches the value of the parameter.

As reference, we compute the answer in 64-bit precision on the CPU. We use a larger grid here, because the differences in memory traffic become more visible on larger problems.

In [ ]:
n_mixed = 2048
U_mixed = Float32.(rand(0.0:0.001:0.1, n_mixed, n_mixed))
V_mixed = zeros(Float32, n_mixed, n_mixed)
mid = div(n_mixed, 2)
U_mixed[mid-5:mid+5, mid-5:mid+5] .= Float32(1.0 * init_multi)
V_mixed[mid-5:mid+5, mid-5:mid+5] .= Float32(0.5 * init_multi)

# reference output in 64-bit precision, computed on the CPU
U_ref64, V_ref64 = Float64.(U_mixed), Float64.(V_mixed)
gs_gpu_kernel!(CPU(), (16, 16))(U_ref64, V_ref64, Float64.(U_mixed), Float64.(V_mixed),
    0.16, 0.08, 0.035, 0.06, 1.0, 1.0, n_mixed, n_mixed, 0.0, ndrange=(n_mixed, n_mixed))

# every array argument is selected by the value of `storage_bits`
storage(A) = KernelTuner.Tunable("storage_bits", Dict(32 => Float32.(A), 16 => Float16.(A)))
arguments_mixed = [storage(U_mixed), storage(V_mixed), storage(U_mixed), storage(V_mixed),
                   0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n_mixed, n_mixed, 0.0f0]

To measure the accuracy, we add an `AccuracyObserver`. It compares the outputs of each configuration to the `answer` and adds the error, here the root-mean-square error (RMSE), to the results as `error`. Instead of a name such as `"RMSE"`, `"MAE"` or `"relative error"`, you can also pass your own Julia function `(answer, output) -> error`. With an accuracy observer, configurations whose output deviates from the answer are no longer rejected as in Section 4.1: instead, the deviation is measured.

In [ ]:
tune_params_mixed = [
    ("block_size_x", [8, 16, 32, 64, 128]),  # workgroup size along x
    ("block_size_y", [1, 2, 4, 8, 16]),      # workgroup size along y
    ("storage_bits", [32, 16]),     # precision of the arrays
    ("compute_bits", [32, 16]),     # precision of the arithmetic
]

accuracy_observer = AccuracyObserver("RMSE")   # or your own function, e.g. AccuracyObserver((answer, output) -> maximum(abs.(answer .- output)))

results_mixed, _ = KernelTuner.tune_kernel(
    "gs_gpu_kernel_mixed!",
    get_cell_by_tag("gs_gpu_kernel_mixed"),
    (n_mixed, n_mixed),
    arguments_mixed,
    tune_params_mixed,
    restrictions=restrictions,
    answer=[U_ref64, V_ref64, fill(nothing, length(arguments_mixed) - 2)...],
    observers=[accuracy_observer],
    verbose=true,
)

Now we can choose the fastest configuration whose error stays within a tolerance of our choice. `Float32` has a relative precision of about $10^{-7}$ and `Float16` of about $10^{-3}$, so with values of $U$ and $V$ below 1, a tolerance of $10^{-6}$ separates the two.

In [ ]:
max_error = 1e-6
precision_label(r) = "storage $(r["storage_bits"])-bit, compute $(r["compute_bits"])-bit"
config_string_mixed(r) = join(["$k=$(r[k])" for k in first.(tune_params_mixed)], ", ")

fastest_overall = argmin(r -> r["time"], results_mixed)
fastest_accurate = argmin(r -> r["time"], filter(r -> r["error"] <= max_error, results_mixed))
for (name, r) in [("Fastest overall", fastest_overall), ("Fastest with error ≤ $max_error", fastest_accurate)]
    println(rpad(name * ":", 32), config_string_mixed(r), "  ($(round(r["time"], digits=3)) ms, error $(round(r["error"], sigdigits=3)))")
end

scatter([r["error"] for r in results_mixed], [r["time"] for r in results_mixed]; group=precision_label.(results_mixed),
    markersize=5, markerstrokewidth=0.3, markerstrokecolor=:gray40, xscale=:log10,
    xlabel="Error (RMSE)", ylabel="Time (ms)", title="Accuracy versus time", legend=:outertopright,
    size=(900, 400), dpi=200, left_margin=3Plots.mm, bottom_margin=3Plots.mm)
vline!([max_error]; linestyle=:dash, color=:gray, label="maximum error")

You should see the configurations grouped by precision: with 32-bit storage and arithmetic, the result is almost exact, while 16-bit storage or arithmetic increases the error by several orders of magnitude. Whether the 16-bit configurations are also faster depends on your GPU and on how memory-bound the kernel is. Whether the extra error is acceptable depends on your application, which is why we choose the fastest configuration within a tolerance. Keep in mind that this is the error of a single time step: over a full simulation, errors accumulate.

## 4.4. User-Defined Metrics

Execution time alone does not tell you how close a kernel comes to the capabilities of the hardware. With `metrics`, Kernel Tuner computes derived quantities for every configuration, such as the achieved computational throughput (GFLOP/s) and memory bandwidth (GB/s). A metric is either a string with an expression, in which the tunable parameters and measured results such as `time` (in milliseconds) can be used, or a Julia function that receives the parameters and results of a configuration as a `Dict`. Metrics are evaluated in order and can use earlier metrics, so we pass them as a vector of `name => metric` pairs, which keeps that order.

For each grid point, the kernel performs 34 floating-point operations: 9 for each of the two Laplacians, 2 for $UV^2$, 5 each for `DU` and `DV`, and 2 each for the new values of `u` and `v`. At the very least, it reads `U` and `V` and writes `Uout` and `Vout` once: 4 × 4 bytes. This is a lower bound, which assumes that the reads of the neighbors are served from cache. Any metric can also be used as the `objective`, here we maximize the bandwidth.

In [ ]:
flops_per_point = 34                    # floating-point operations per grid point, counted from the kernel
bytes_per_point = 4 * sizeof(Float32)   # read U and V, write Uout and Vout
grid_points = (n - 2)^2                 # the boundary is not updated

metrics = [
    "GFLOP/s" => "$(flops_per_point * grid_points / 1e9) / (time / 1e3)",   # time is in milliseconds
    "GB/s" => "$(bytes_per_point * grid_points / 1e9) / (time / 1e3)",
    "work-items" => p -> p["block_size_x"] * p["block_size_y"],            # a Julia function works too
]

tune_params_medium = [
    ("P", [1, 2, 4, 8]),                        # tile size per work-item along y
    ("Q", [1, 2, 4, 8]),                        # tile size per work-item along x
    ("block_size_x", [8, 16, 32, 64, 128]),     # workgroup size along x
    ("block_size_y", [1, 2, 4, 8, 16]),         # workgroup size along y
]

results_metrics, env_metrics = KernelTuner.tune_kernel(
    "gs_gpu_kernel_tiled!",
    get_cell_by_tag("gs_gpu_kernel_tiled"),
    (n, n),
    arguments,
    tune_params_medium,
    restrictions=restrictions,
    grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
    metrics=metrics,
    objective="GB/s",               # any metric can be the objective
    objective_higher_is_better=true,
    verbose=true,
)

The heatmap function from the setup also accepts any metric. Let's look at the achieved bandwidth for the tile sizes, using the workgroup size of the best configuration:

In [ ]:
best = env_metrics["best_config"]
results_best_wg = filter(r -> r["block_size_x"] == best["block_size_x"] && r["block_size_y"] == best["block_size_y"], results_metrics)
plot_tuning_heatmap(results_best_wg, "P", "Q"; objective="GB/s", higher_is_better=true)

Compare the achieved bandwidth with the peak memory bandwidth of your GPU, which you can find in its specifications. If the best configuration stays far below the peak, the kernel is not limited by memory bandwidth. For a grid of 1024 × 1024 points, the kernel runs for only a fraction of a millisecond, which may not be enough to fully occupy the GPU. Recall that Kernel Tuner measures the time the kernel runs on the GPU: in the simulation, the time to launch each kernel from the host comes on top of that, and for a kernel this short it can be as large as the kernel itself. A larger grid or several time steps per kernel launch then helps more than further tuning.

And the other way around, with `GFLOP/s` over the workgroup sizes given the best `P` and `Q`: 

In [ ]:
best = env_metrics["best_config"]
results_best_wg = filter(r -> r["P"] == best["P"] && r["Q"] == best["Q"], results_metrics)
plot_tuning_heatmap(results_best_wg, "block_size_x", "block_size_y"; objective="GFLOP/s", higher_is_better=true)

Metrics can also use the measurements of observers. For example, to compute the energy efficiency in GFLOP/J from the energy measured in Section 4.2, pass the energy observer and add a metric that divides the work by the measured energy:

```julia
metrics_energy = ["GFLOP/J" => "$(flops_per_point * grid_points / 1e9) / $energy_key"]
# KernelTuner.tune_kernel(...; observers=[energy_observer], metrics=metrics_energy, objective="GFLOP/J", objective_higher_is_better=true)
```

# 5. All Combined

In Section 3.3 we tuned the workgroup and tile sizes together on the large search space, and in Section 4.2 we measured the energy on a small one. Let's combine the various concepts we've explored: we tune the large search space of Section 3.3 for energy with output verification, while a live plot shows the time and energy of each configuration as it is evaluated.

There is rarely a single best configuration for both time and energy. The black line in the plot is the *Pareto front*: the configurations for which no other configuration is both faster and uses less energy. Every configuration on the front is a sensible choice, and which one is best depends on how much you value time versus energy.

Measuring energy makes each configuration take longer to evaluate, as the energy observers measure each configuration for about a second. That's why we again use the optimization strategy and limits from Sections 3.2 and 3.3. We use a separate cache file, as the results cached in Section 3.3 contain no energy measurements. Configurations that are already in the cache are not measured again, so running the cell again adds new configurations to the plot.

In [ ]:
# the same observer as in Section 4.2; the name of the energy field differs per vendor
energy_observer, energy_fields = KernelTuner.get_energy_observer_for_backend("default")
energy_key = only(filter(endswith("energy"), energy_fields))

# reference output: one step of the simple kernel
U_ref, V_ref = copy(U), copy(V)
gs_gpu_kernel!(CPU(), (16, 16))(U_ref, V_ref, U, V, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, n, n, 0.0f0, ndrange=(n, n))

# one entry per kernel argument: check Uout and Vout, skip the rest
answer = [U_ref, V_ref, fill(nothing, length(arguments) - 2)...]

cachefile_energy = "gs_gpu_kernel_tuned_energy.json"

# the plot of time versus energy is redrawn as configurations are added to the cache file
results_combined, env_combined = with_live_tuning_plot(cachefile_energy; plotter=plot_time_energy, energy_key=energy_key) do
    KernelTuner.tune_kernel(
        "gs_gpu_kernel_tiled!",
        get_cell_by_tag("gs_gpu_kernel_tiled"),
        (n, n),
        arguments,
        tune_params,                    # the large search space of Section 3.3
        restrictions=restrictions,
        grid_div_x=("block_size_x", "Q"), grid_div_y=("block_size_y", "P"),
        observers=[energy_observer],
        objective=energy_key,           # the strategy minimizes energy, the plot shows both time and energy
        objective_higher_is_better=false,   # lower energy is better
        cache=cachefile_energy,
        answer=answer,
        atol=1e-5,  # maximum absolute difference allowed, the GPU and CPU may round slightly differently
        verbose=true,
    )
end

# if you'd like to start over without the cached results, uncomment the following line
# rm(cachefile_energy)

Finally, let's list the configurations on the Pareto front, from fastest to most energy-efficient:

In [ ]:
measured(r) = all(k -> get(r, k, nothing) isa Real && r[k] > 0, ("time", energy_key))   # skip failed configurations
results_measured = filter(measured, results_combined)
front = pareto_front([r["time"] for r in results_measured], [r[energy_key] for r in results_measured])   # from KernelTuner.jl

most_performant = first(results_measured[front])
for r in results_measured[front]
    params = join(["$k=$(r[k])" for k in first.(tune_params)], ", ")
    time_saved = 100 * (most_performant["time"] - r["time"]) / most_performant["time"]
    energy_saved = 100 * (most_performant[energy_key] - r[energy_key]) / most_performant[energy_key]
    if r == most_performant
        savings_string = ""
    else
        savings_string = time_saved > 0 ? "saves $(round(time_saved, sigdigits=3))% time" : "trades $(round(-time_saved, sigdigits=3))% time"
        savings_string *= energy_saved > 0 ? " to save $(round(energy_saved, sigdigits=3))% energy" : " by trading $(round(-energy_saved, sigdigits=3))% energy"
    end
    println(params, "  ($(round(r["time"], digits=3)) ms, $(round(r[energy_key], sigdigits=3)) J); $savings_string")
end

If you are limited by the time to solution, pick the configuration at the fast end of the front; if you are limited by an energy or power budget, pick one further along. Configurations far above the front are worse in both respects and can be discarded. Keep in mind that, like the timings, energy measurements are noisy, especially for kernels that take well under a millisecond.

# Summary

In this notebook, you have:
- made the parameters of a KernelAbstractions kernel tunable with `Val` arguments, and tuned the workgroup size and the tile size;
- inspected and visualized the tuning results;
- used restrictions, optimization strategies, limits and caching to tune larger search spaces;
- verified the output of every configuration, measured energy, traded accuracy for speed with mixed precision, and defined your own metrics;
- chosen a configuration from the Pareto front of time and energy.

The same approach works for any KernelAbstractions kernel: make the values you want to tune `Val` arguments, list them in `tune_params`, and let Kernel Tuner find the best configuration for your GPU. Remember to tune again when you move to a different GPU or problem size!

**Further reading**
- Kernel Tuner documentation: https://kerneltuner.github.io/kernel_tuner/stable/
- Kernel Tuner on GitHub: https://github.com/KernelTuner/kernel_tuner
- KernelTuner.jl on GitHub: https://github.com/KernelTuner/KernelTuner.jl